# Nifty 50 Historical Data Downloader

**Important Note**: Flattrade API's `get_time_price_series` only provides intraday data for the **current trading session**, not historical multi-year data. For 10 years of historical data, we have two options:

1. **Daily OHLC Data** (recommended) - Works via `get_daily_price_series`
2. **Live Data Recorder** - Run during market hours to collect 1-min data going forward

This notebook implements **Option 1: Daily Historical Data Download** which is immediately available.

In [2]:
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd

workspace_root = Path.cwd().resolve()
experiment_root = workspace_root / "scripts" / "claude" / "experiment8"
api_root = experiment_root / "pythonAPI-main" / "pythonAPI-main"
sys.path.insert(0, str(experiment_root))
sys.path.insert(0, str(api_root))
sys.path.insert(0, str(api_root / "dist"))

# Flattrade credentials
USER_ID = "FZ31397"
USER_TOKEN = "890365fb384375e17383609e7d0b033e3591add7409dbe3e281dd2f992e8fc26"

NOTEBOOK_DIR = (workspace_root / "scripts" / "github_copilot").resolve()
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

YEARS_OF_HISTORY = 10
print(f"📁 Output directory: {NOTEBOOK_DIR}")
print(f"📅 Data range: Last {YEARS_OF_HISTORY} years")

📁 Output directory: D:\StockMarket\StockMarket\scripts\github_copilot\scripts\github_copilot
📅 Data range: Last 10 years


In [3]:
from api_helper import NorenApiPy

def initialize_api():
    if not USER_ID or not USER_TOKEN:
        raise RuntimeError("Set USER_ID and USER_TOKEN in this notebook")
    client = NorenApiPy()
    success = client.set_session(userid=USER_ID, password="", usertoken=USER_TOKEN)
    if not success:
        raise RuntimeError("Flattrade session setup failed")
    print(f"✅ Logged in as {USER_ID}")
    return client

api = initialize_api()

✅ Logged in as FZ31397


## Option 1: Download Daily OHLC Data (10 Years)

This works immediately and provides daily candles with Open, High, Low, Close, Volume.

In [4]:
def download_daily_data(symbol="Nifty 50", years=10):
    """Download daily OHLC data for specified years."""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * years)
    
    print(f"📊 Downloading {symbol} daily data...")
    print(f"   From: {start_date.date()}")
    print(f"   To: {end_date.date()}")
    
    try:
        response = api.get_daily_price_series(
            exchange="NSE",
            tradingsymbol=symbol,
            startdate=int(start_date.timestamp()),
            enddate=int(end_date.timestamp())
        )
        
        if not response or not isinstance(response, list):
            print(f"❌ No data returned. Trying alternate symbol names...")
            
            # Try different symbol names
            for alt_symbol in ["NIFTY 50", "Nifty50", "NIFTY50", "NIFTY"]:
                print(f"   Trying: {alt_symbol}")
                response = api.get_daily_price_series(
                    exchange="NSE",
                    tradingsymbol=alt_symbol,
                    startdate=int(start_date.timestamp()),
                    enddate=int(end_date.timestamp())
                )
                if response and isinstance(response, list):
                    print(f"   ✅ Found data with symbol: {alt_symbol}")
                    break
        
        if response and isinstance(response, list):
            df = pd.DataFrame(response)
            print(f"\n✅ Downloaded {len(df)} daily candles")
            
            # Parse and format data
            if 'time' in df.columns:
                df['time'] = pd.to_datetime(df['time'], errors='coerce')
                df = df.sort_values('time')
            
            # Rename columns for clarity
            column_mapping = {
                'into': 'open',
                'inth': 'high', 
                'intl': 'low',
                'intc': 'close',
                'intv': 'volume'
            }
            df = df.rename(columns=column_mapping)
            
            # Save to CSV
            output_file = NOTEBOOK_DIR / f"nifty50_daily_{years}years.csv"
            df.to_csv(output_file, index=False)
            print(f"💾 Saved to: {output_file}")
            
            print(f"\n📈 Data Summary:")
            print(f"   Columns: {df.columns.tolist()}")
            print(f"   Date range: {df['time'].min()} to {df['time'].max()}")
            print(f"\n   First 5 rows:")
            print(df.head())
            print(f"\n   Last 5 rows:")
            print(df.tail())
            
            return df
        else:
            print("❌ Unable to fetch daily data with any symbol name")
            return None
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Download the data
daily_df = download_daily_data(symbol="Nifty 50", years=YEARS_OF_HISTORY)

📊 Downloading Nifty 50 daily data...
   From: 2016-02-05
   To: 2026-02-02
❌ No data returned. Trying alternate symbol names...
   Trying: NIFTY 50
   Trying: Nifty50
   Trying: NIFTY50
   Trying: NIFTY
❌ Unable to fetch daily data with any symbol name


## Option 2: Live Intraday Data Recorder

Since historical intraday data isn't available via API, here's a recorder that collects live 1-minute data during market hours. Run this daily to build historical dataset over time.

In [5]:
def record_todays_intraday_data():
    """Record today's 1-minute data (only works during/after market hours)."""
    
    SPOT_TOKEN = "26000"  # From found_symbols.csv
    today = datetime.now()
    
    print(f"📡 Attempting to fetch today's intraday data ({today.date()})...")
    
    try:
        # Try without time constraints (gets today's data by default)
        response = api.get_time_price_series(
            exchange="NSE",
            token=SPOT_TOKEN,
            starttime=None,  # Defaults to today 00:00
            endtime=None,
            interval=1  # 1 minute
        )
        
        if response and isinstance(response, list) and len(response) > 0:
            df = pd.DataFrame(response)
            
            if 'time' in df.columns:
                df['time'] = pd.to_datetime(df['time'], dayfirst=True)
            
            # Rename columns
            column_mapping = {
                'into': 'open',
                'inth': 'high',
                'intl': 'low', 
                'intc': 'close',
                'intv': 'volume',
                'intvwap': 'vwap'
            }
            df = df.rename(columns=column_mapping)
            
            # Save with date stamp
            date_str = today.strftime('%Y%m%d')
            output_file = NOTEBOOK_DIR / f"nifty50_1min_{date_str}.csv"
            df.to_csv(output_file, index=False)
            
            print(f"\n✅ Recorded {len(df)} 1-minute candles for {today.date()}")
            print(f"💾 Saved to: {output_file}")
            print(f"\n   Time range: {df['time'].min()} to {df['time'].max()}")
            print(f"\n   Sample data:")
            print(df.head(10))
            
            return df
        else:
            print("❌ No intraday data available yet.")
            print("\n💡 Possible reasons:")
            print("   1. Market hasn't opened yet (opens at 9:15 AM IST)")
            print("   2. Market is closed (weekend/holiday)")
            print("   3. Data only available during/after trading hours")
            print("\n📌 Run this cell during market hours (9:15 AM - 3:30 PM IST)")
            print("   or after market close to get today's complete data.")
            return None
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Try to record today's data
intraday_df = record_todays_intraday_data()

📡 Attempting to fetch today's intraday data (2026-02-02)...
❌ No intraday data available yet.

💡 Possible reasons:
   1. Market hasn't opened yet (opens at 9:15 AM IST)
   2. Market is closed (weekend/holiday)
   3. Data only available during/after trading hours

📌 Run this cell during market hours (9:15 AM - 3:30 PM IST)
   or after market close to get today's complete data.


## Summary and Next Steps

### What Works:
- ✅ **Daily OHLC data** for 10 years (if API supports it)
- ✅ **Today's intraday data** during market hours

### What Doesn't Work:
- ❌ Historical multi-year 1-minute data via Flattrade API (not supported)

### Recommendations:

**For 10 years of historical data:**
1. Use daily data (sufficient for most analysis)
2. Or use alternative data sources:
   - NSE official website (free but manual)
   - Yahoo Finance API (yfinance library)
   - Commercial providers (Zerodha, AlphaVantage)

**For building 1-minute historical dataset:**
1. Run the intraday recorder daily after market close
2. Accumulate data day-by-day
3. After 1 year, you'll have full historical 1-min data

**Quick Alternative - Yahoo Finance:**

In [ ]:
# Alternative: Download using yfinance (free, no authentication needed)
# Uncomment and run if you want immediate historical data

# !pip install yfinance

# import yfinance as yf

# print("📥 Downloading from Yahoo Finance...")
# nifty = yf.download("^NSEI", start="2016-01-01", end="2026-02-02", interval="1d")
# output_file = NOTEBOOK_DIR / "nifty50_10y_yfinance.csv"
# nifty.to_csv(output_file)
# print(f"✅ Saved {len(nifty)} daily candles to {output_file}")
# print(nifty.head())
# print(nifty.tail())

## Working Solution: Yahoo Finance API

Since Flattrade doesn't provide historical data, let's use Yahoo Finance (free, no authentication required):

In [6]:
# Install yfinance if not already installed
import subprocess
import sys

try:
    import yfinance as yf
    print("✅ yfinance already installed")
except ImportError:
    print("📦 Installing yfinance...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance", "-q"])
    import yfinance as yf
    print("✅ yfinance installed successfully")

print(f"\n📥 Downloading Nifty 50 data from Yahoo Finance...")
print(f"   Symbol: ^NSEI (Nifty 50 Index)")
print(f"   Period: Last {YEARS_OF_HISTORY} years")

# Download data
end_date = datetime.now()
start_date = end_date - timedelta(days=365 * YEARS_OF_HISTORY)

nifty_data = yf.download(
    "^NSEI", 
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    interval="1d",
    progress=False
)

if not nifty_data.empty:
    # Save to CSV
    output_file = NOTEBOOK_DIR / f"nifty50_daily_{YEARS_OF_HISTORY}years_yfinance.csv"
    nifty_data.to_csv(output_file)
    
    print(f"\n✅ Successfully downloaded {len(nifty_data)} daily candles")
    print(f"💾 Saved to: {output_file}")
    print(f"\n📊 Data Summary:")
    print(f"   Columns: {nifty_data.columns.tolist()}")
    print(f"   Date range: {nifty_data.index.min()} to {nifty_data.index.max()}")
    print(f"\n   First 10 rows:")
    print(nifty_data.head(10))
    print(f"\n   Last 10 rows:")
    print(nifty_data.tail(10))
    print(f"\n   Statistics:")
    print(nifty_data.describe())
else:
    print("❌ Failed to download data from Yahoo Finance")

📦 Installing yfinance...
✅ yfinance installed successfully

📥 Downloading Nifty 50 data from Yahoo Finance...
   Symbol: ^NSEI (Nifty 50 Index)
   Period: Last 10 years

✅ Successfully downloaded 2461 daily candles
💾 Saved to: D:\StockMarket\StockMarket\scripts\github_copilot\scripts\github_copilot\nifty50_daily_10years_yfinance.csv

📊 Data Summary:
   Columns: [('Close', '^NSEI'), ('High', '^NSEI'), ('Low', '^NSEI'), ('Open', '^NSEI'), ('Volume', '^NSEI')]
   Date range: 2016-02-05 00:00:00 to 2026-01-30 00:00:00

   First 10 rows:
Price             Close         High          Low         Open  Volume
Ticker            ^NSEI        ^NSEI        ^NSEI        ^NSEI   ^NSEI
Date                                                                  
2016-02-05  7489.100098  7503.149902  7406.649902  7418.250000  249800
2016-02-08  7387.250000  7512.549805  7363.200195  7489.700195  171500
2016-02-09  7298.200195  7323.450195  7275.149902  7303.950195  212100
2016-02-10  7215.700195  7271.85009